In [1]:
# Code to map Maven elements to Origami's HealthcareUtilization elements
# Using semantic similarity library
# Author: Sandeep Chintabathina
# April 2026

In [2]:
# For each Maven data element, check if there is an exact or close matching data element in the origami HealthcareUtilization package
# The output will list the maven element alongside the origami element and other supporting attributes

In [3]:
from sentence_transformers import SentenceTransformer


In [4]:
import pandas as pd
import numpy as np

In [5]:
# Read maven elements
df_maven = pd.read_csv('output/all_db_elements_use_this.csv',na_filter=False)
len(df_maven)

6232

In [6]:
df_maven.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6232 entries, 0 to 6231
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   QUESTION_ID     6232 non-null   object
 1   value           6232 non-null   object
 2   QUESTIONSET_ID  6232 non-null   object
 3   MODEL_NAME      6232 non-null   object
dtypes: object(4)
memory usage: 194.9+ KB


In [7]:
def clean_up(name):
    new_name =''
    for ch in name:
        # If space, digits, uppercase letters, lowercase letters, or underscore, add it to new name otherwise ignore
        if (ord(ch)==32 or 48<=ord(ch)<=57 or 65<=ord(ch)<=90 or 97<=ord(ch)<=122 or ord(ch)==95 ):
            new_name+=ch
    return new_name.strip()

In [8]:
clean_up('[sdfs] 34_AS ^#~')

'sdfs 34_AS'

In [10]:
# Clean up questions or data elements
df_maven.loc[:,'QUESTION_ID'] =[ clean_up(q) for q in df_maven.loc[:,'QUESTION_ID']]

In [11]:
df_maven.loc[:,'MODEL_NAME'].value_counts(dropna=False)

MODEL_NAME
DiseaseSurveillanceModel_General        3192
DiseaseSurveillanceModel_STD            1361
DiseaseSurveillanceModel_Hep             754
Childhood_Lead_Child_Model               386
ClusterModel                             205
Childhood_Lead_Investigation_Model       136
DiseaseSurveillanceModel_Coinfection     114
PortalApplicationModel                    41
EHRAndScreeningModel_STD                  22
DiseaseSurveillanceModel_AggNETSS         14
IsolationAndControlModel                   7
Name: count, dtype: int64

In [12]:
# General clean up
df_maven.loc[:,'MODEL_NAME'] = [m.strip() for m in df_maven.loc[:,'MODEL_NAME']]

In [13]:
# creating a sorting order for model name which is then used to sort the dataframe 
# Using the order from value counts above
sorter = ['DiseaseSurveillanceModel_General','DiseaseSurveillanceModel_STD','DiseaseSurveillanceModel_Hep','Childhood_Lead_Child_Model',
          'ClusterModel','Childhood_Lead_Investigation_Model','DiseaseSurveillanceModel_Coinfection','PortalApplicationModel',
          'EHRAndScreeningModel_STD','DiseaseSurveillanceModel_AggNETSS','IsolationAndControlModel']

In [14]:
# convert model_name column to category
df_maven.MODEL_NAME = df_maven.MODEL_NAME.astype('category')
# Set sorter as the categories hierarchy
df_maven.MODEL_NAME = df_maven.MODEL_NAME.cat.set_categories(sorter)


In [15]:
df_maven.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6232 entries, 0 to 6231
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   QUESTION_ID     6232 non-null   object  
 1   value           6232 non-null   object  
 2   QUESTIONSET_ID  6232 non-null   object  
 3   MODEL_NAME      6232 non-null   category
dtypes: category(1), object(3)
memory usage: 152.7+ KB


In [16]:
# Sort df on model_name
df_maven = df_maven.sort_values(['MODEL_NAME'])

In [17]:
df_maven = df_maven.reset_index(drop=True)

In [18]:
df_maven

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME
0,CURRENT_TELEPHONE_WORK,(999) 999-9999,DEMOGRAPHIC,DiseaseSurveillanceModel_General
1,EAT_YOGURT_PURCHASED_FROM,Yogurt Melts,RISK_HISTORY,DiseaseSurveillanceModel_General
2,CDR_PERSON_OR_AGENCY_REPORTING,Zyan Paresa,PORTAL_CDR_CASE_SUBMISSION,DiseaseSurveillanceModel_General
3,ANTACID_EXPOSURE,UNKNOWN,CLINICAL,DiseaseSurveillanceModel_General
4,DIARY_DAY_1_DINNER,Zippys -No Bean Chili,RISK_HISTORY,DiseaseSurveillanceModel_General
...,...,...,...,...
6227,PROGRAM_CODE,STD,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
6228,PRODUCT_DESCRIPTION,YERSINIOSIS; Yersinia enterocolitica; psudotob...,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
6229,REPORT_CATEGORY_LAB,Urgent - upon order,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
6230,REPORT_CATEGORY_PROVIDER,Urgent,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel


In [19]:
# duplicates
df_maven[df_maven.duplicated(['QUESTION_ID'],keep=False)]

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME
0,CURRENT_TELEPHONE_WORK,(999) 999-9999,DEMOGRAPHIC,DiseaseSurveillanceModel_General
2,CDR_PERSON_OR_AGENCY_REPORTING,Zyan Paresa,PORTAL_CDR_CASE_SUBMISSION,DiseaseSurveillanceModel_General
10,CASE_REFERRED,YES,ADMINISTRATIVE,DiseaseSurveillanceModel_General
12,PRODUCT_DESCRIPTION,YERSINIOSIS; Yersinia enterocolitica; psudotob...,PORTAL_CDR_CASE_SUBMISSION,DiseaseSurveillanceModel_General
17,REPORTING_SOURCE_TYPE,SELF,ADMINISTRATIVE,DiseaseSurveillanceModel_General
...,...,...,...,...
6222,EPISODE_DATE,12/31/2016,ADMINISTRATIVE,DiseaseSurveillanceModel_AggNETSS
6224,MMWR_WEEK,53,ADMINISTRATIVE,DiseaseSurveillanceModel_AggNETSS
6226,ISO_QUAR_CONTACT_INSTRUCTIONS,"Surveillance, testing, counseling, interview, ...",ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
6228,PRODUCT_DESCRIPTION,YERSINIOSIS; Yersinia enterocolitica; psudotob...,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel


In [20]:
# Keeping the first appearance of question
df_maven = df_maven.drop_duplicates(['QUESTION_ID'])

In [22]:
df_maven[df_maven.duplicated(['QUESTION_ID'],keep=False)]

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME


In [23]:
df_maven = df_maven.reset_index(drop=True)

In [24]:
df_maven

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME
0,CURRENT_TELEPHONE_WORK,(999) 999-9999,DEMOGRAPHIC,DiseaseSurveillanceModel_General
1,EAT_YOGURT_PURCHASED_FROM,Yogurt Melts,RISK_HISTORY,DiseaseSurveillanceModel_General
2,CDR_PERSON_OR_AGENCY_REPORTING,Zyan Paresa,PORTAL_CDR_CASE_SUBMISSION,DiseaseSurveillanceModel_General
3,ANTACID_EXPOSURE,UNKNOWN,CLINICAL,DiseaseSurveillanceModel_General
4,DIARY_DAY_1_DINNER,Zippys -No Bean Chili,RISK_HISTORY,DiseaseSurveillanceModel_General
...,...,...,...,...
5439,COMMENTS,whatever,ADMINISTRATIVE,DiseaseSurveillanceModel_AggNETSS
5440,PRODUCT_CODE,YER,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
5441,PROGRAM_CODE,STD,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
5442,REPORT_CATEGORY_LAB,Urgent - upon order,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel


In [25]:
# HealthcareUtilization elements
df_hu = pd.read_csv('origami/csv/Custom__HealthcareUtilization.csv',na_filter=False)
len(df_hu)

430

In [26]:
df_hu.loc[:,'Label'].value_counts(dropna=False)

Label
                                             375
Healthcare Utilization ID                      1
BED_BOUND_REQUIRE_HELP_SPECIFY                 1
Standard Precaution End Date                   1
Other Precaution Start Date                    1
Other Precaution End Date                      1
Roommate Date of birth                         1
ADMISSION_DATE                                 1
DISCHARGE_DATE                                 1
Roommate Date of birth 2                       1
Roommate Date of birth 3                       1
PRECAUTIONS_FOLLOWED                           1
CAREGIVER_PPE                                  1
AFIM_NA_DL_PPE                                 1
Days shared same room                          1
Enhanced Precaution End Date                   1
Facility Street Address                        1
Setting Other Specify                          1
Roommate Name/surname                          1
Cleaning supplies                              1
FACILITY_NAME 

In [27]:
# Get the two columns of interest and non-blank values
df_hu = df_hu[['Field','Label']]
df_hu = df_hu[df_hu['Label']!='']
len(df_hu)

55

In [28]:
df_hu = df_hu.reset_index(drop=True)

In [29]:
df_hu

,Field,Label
0,HealthcareUtilizationID,Healthcare Utilization ID
1,ClientID,Client
2,EntryUserID,Entry User
3,EntryDate,Entry Date
4,ModifiedUserID,Modified User
5,ModifiedDate,Modified Date
6,IncidentID,Incident
7,CustomCode1ID,Shared medical equipment
8,CustomCode2ID,Shared bathrooms
9,CustomCode3ID,Roommate count


In [30]:
# Original label to be used later for output
df_hu.rename(columns={'Label':'original_label'},inplace=True)
df_hu.columns

Index(['Field', 'original_label'], dtype='object')

In [31]:
# Remove only the first occurrence of [] in a string
def remove_squares(somestring):
    if ']' in somestring:
        for i in range(len(somestring)):
            if somestring[i]==']':
                return somestring[i+1:].strip()
    return somestring.strip()


In [32]:
remove_squares('asd[asdasd]asc[obas]da')

'asc[obas]da'

In [33]:
# Process the column values (eliminate the values in square brackets)
df_hu.loc[:,'Label'] = [ remove_squares(label) for label in df_hu.loc[:,'original_label']]

In [34]:
# Clean up column names in both data sets
df_hu.loc[:,'Label'] =[ clean_up(q) for q in df_hu.loc[:,'Label']]

In [35]:
df_hu.loc[:,'Label'].value_counts()

Label
Healthcare Utilization ID          1
BED_BOUND_REQUIRE_HELP_SPECIFY     1
Standard Precaution End Date       1
Other Precaution Start Date        1
Other Precaution End Date          1
Roommate Date of birth             1
ADMISSION_DATE                     1
DISCHARGE_DATE                     1
Roommate Date of birth 2           1
Roommate Date of birth 3           1
PRECAUTIONS_FOLLOWED               1
CAREGIVER_PPE                      1
AFIM_NA_DL_PPE                     1
Days shared same room              1
Enhanced Precaution End Date       1
Facility Street Address            1
Setting Other Specify              1
Roommate Namesurname               1
Cleaning supplies                  1
FACILITY_NAME                      1
FACILITY_LOCATION                  1
Other longterm care type           1
HOSPITALIZED_TRANSFER_LOCATION     1
Roommate Namesurname 2             1
Roommate Namesurname 3             1
PRECAUTIONS_FOLLOWED_OTHER         1
Standard Precaution Start Date  

In [36]:
# duplicates labels do not exist 
df_hu[df_hu.duplicated(['Label'],keep=False)]

,Field,original_label,Label


In [79]:
# dropping them
#df_hai = df_hai.drop_duplicates(['Label'])

In [80]:
#df_hai[df_hai.duplicated(['Label'],keep=False)]

In [37]:
df_hu = df_hu.reset_index(drop=True)
df_hu

,Field,original_label,Label
0,HealthcareUtilizationID,Healthcare Utilization ID,Healthcare Utilization ID
1,ClientID,Client,Client
2,EntryUserID,Entry User,Entry User
3,EntryDate,Entry Date,Entry Date
4,ModifiedUserID,Modified User,Modified User
5,ModifiedDate,Modified Date,Modified Date
6,IncidentID,Incident,Incident
7,CustomCode1ID,Shared medical equipment,Shared medical equipment
8,CustomCode2ID,Shared bathrooms,Shared bathrooms
9,CustomCode3ID,Roommate count,Roommate count


In [82]:
# Do a semantic match between Maven elements and Origami HAI labels

In [39]:
maven_elements = [str.replace(maven_de,'_',' ') for maven_de in df_maven.QUESTION_ID]
origami_elements = [str.replace(col,'_',' ') for col in df_hu.Label]

In [40]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [41]:
# Compute embeddings
embeddings_maven = model.encode(maven_elements)
embeddings_origami = model.encode(origami_elements)

In [42]:
print(len(embeddings_origami))
print(len(embeddings_maven))
embeddings_origami
embeddings_maven

55
5444


array([[-0.16024935,  0.0271387 ,  0.06354488, ..., -0.00712903,
        -0.01673306,  0.06124594],
       [ 0.04245438,  0.03854417,  0.0684232 , ..., -0.06141916,
         0.08061311, -0.07027474],
       [-0.08168999,  0.03473966, -0.06668424, ..., -0.04572672,
        -0.00667135,  0.0398071 ],
       ...,
       [-0.06123391,  0.04124329, -0.09608547, ...,  0.11074385,
         0.03158536, -0.0520657 ],
       [ 0.03361172,  0.06674883, -0.07311527, ...,  0.01481686,
         0.04655252, -0.02631201],
       [-0.00503135,  0.02209355, -0.08806273, ..., -0.04621199,
         0.04993448,  0.04251292]], dtype=float32)

In [43]:
# Compute cosine similarities
similarities = model.similarity(embeddings_maven,embeddings_origami)

In [44]:
similarities

tensor([[ 0.1693,  0.1604,  0.0948,  ...,  0.0879,  0.0284,  0.0743],
        [-0.0171,  0.0250,  0.0455,  ...,  0.0533,  0.1268,  0.1232],
        [ 0.1585,  0.2240,  0.2282,  ...,  0.1334,  0.0590,  0.1924],
        ...,
        [ 0.1240,  0.2358,  0.2094,  ...,  0.1223, -0.0118,  0.1101],
        [ 0.1567,  0.0612,  0.0157,  ...,  0.0683,  0.0801,  0.0952],
        [ 0.1452,  0.1426,  0.0463,  ...,  0.0896,  0.0377,  0.1754]])

In [45]:
print(type(similarities))
len(similarities)

<class 'torch.Tensor'>


5444

In [46]:
df_similarities = pd.DataFrame(similarities,index=df_maven.QUESTION_ID,columns=df_hu.original_label)

In [47]:
df_similarities.to_csv('output/similarity_scores_HealthcareUtilization.csv')

In [48]:
df_maven = df_maven.reset_index(drop=True)
df_maven.columns

Index(['QUESTION_ID', 'value', 'QUESTIONSET_ID', 'MODEL_NAME'], dtype='object')

In [49]:
df_output = df_maven[['MODEL_NAME','QUESTION_ID','QUESTIONSET_ID']]

In [94]:
#df_similarities.info()

In [50]:
#For every maven element (index), identify origami elements (column) that is the best match

filtered=[]
for idx in df_similarities.index:
    # Looking for columns with score higher than 0.75
    v = df_similarities.loc[idx,:]>0.75

    # Get the column with the maximum score
    max=0.0
    max_col=''
    for i,col in enumerate(df_similarities.columns):
        if v.iloc[i]==True:
            #print('value',df_similarities.loc[idx,col])
            if (df_similarities.loc[idx,col] > max):
                max = df_similarities.loc[idx,col]
                max_col=col

    #filtered.append({col:df_similarities.loc[idx,col]})
    #print(idx+','+max_col+':'+str(max)+'\n')
    filtered.append({'QUESTION_ID':idx,'target_label':max_col})



In [51]:
df_filtered = pd.DataFrame(filtered)
print(len(df_filtered))
#df_filtered.loc[:,'target_label'].value_counts(dropna=False)

5444


In [52]:
df_merge = pd.merge(df_filtered,df_hu,left_on='target_label',right_on='original_label',how='left')

In [53]:
df_merge

,QUESTION_ID,target_label,Field,original_label,Label
0,CURRENT_TELEPHONE_WORK,,NaN,NaN,NaN
1,EAT_YOGURT_PURCHASED_FROM,,NaN,NaN,NaN
2,CDR_PERSON_OR_AGENCY_REPORTING,,NaN,NaN,NaN
3,ANTACID_EXPOSURE,,NaN,NaN,NaN
4,DIARY_DAY_1_DINNER,,NaN,NaN,NaN
...,...,...,...,...,...
5439,COMMENTS,,NaN,NaN,NaN
5440,PRODUCT_CODE,,NaN,NaN,NaN
5441,PROGRAM_CODE,,NaN,NaN,NaN
5442,REPORT_CATEGORY_LAB,,NaN,NaN,NaN


In [54]:
# merge the above merged df with output dataframe
df_final = pd.merge(df_output,df_merge,on='QUESTION_ID')
#df_final

In [55]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5444 entries, 0 to 5443
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   MODEL_NAME      5444 non-null   category
 1   QUESTION_ID     5444 non-null   object  
 2   QUESTIONSET_ID  5444 non-null   object  
 3   target_label    5444 non-null   object  
 4   Field           18 non-null     object  
 5   original_label  18 non-null     object  
 6   Label           18 non-null     object  
dtypes: category(1), object(6)
memory usage: 261.0+ KB


In [56]:
# Add, rename, remove columns as needed
df_final['target_table'] = ['' if f=='' else 'Custom__HealthcareUtilization' for f in df_final.target_label]
df_final=df_final.rename(columns={'Field':'target_field'})
df_final=df_final.drop('Label',axis=1)


In [57]:
df_final = df_final[["MODEL_NAME",'QUESTION_ID','target_table','target_field','target_label']]
#df_final

In [58]:
df_final

,MODEL_NAME,QUESTION_ID,target_table,target_field,target_label
0,DiseaseSurveillanceModel_General,CURRENT_TELEPHONE_WORK,,NaN,
1,DiseaseSurveillanceModel_General,EAT_YOGURT_PURCHASED_FROM,,NaN,
2,DiseaseSurveillanceModel_General,CDR_PERSON_OR_AGENCY_REPORTING,,NaN,
3,DiseaseSurveillanceModel_General,ANTACID_EXPOSURE,,NaN,
4,DiseaseSurveillanceModel_General,DIARY_DAY_1_DINNER,,NaN,
...,...,...,...,...,...
5439,DiseaseSurveillanceModel_AggNETSS,COMMENTS,,NaN,
5440,IsolationAndControlModel,PRODUCT_CODE,,NaN,
5441,IsolationAndControlModel,PROGRAM_CODE,,NaN,
5442,IsolationAndControlModel,REPORT_CATEGORY_LAB,,NaN,


In [59]:
df_final.loc[:,'MODEL_NAME'].value_counts(dropna=False)

MODEL_NAME
DiseaseSurveillanceModel_General        3192
DiseaseSurveillanceModel_STD            1221
Childhood_Lead_Child_Model               361
DiseaseSurveillanceModel_Hep             296
ClusterModel                             195
Childhood_Lead_Investigation_Model       125
PortalApplicationModel                    30
DiseaseSurveillanceModel_Coinfection      10
EHRAndScreeningModel_STD                   8
IsolationAndControlModel                   4
DiseaseSurveillanceModel_AggNETSS          2
Name: count, dtype: int64

In [60]:
# Convert category type back to str object
df_final.MODEL_NAME = df_final.MODEL_NAME.astype('str')

In [61]:
# Rename model names
df_final.loc[:,'MODEL_NAME'] = [ m.split('_')[1] if 'DiseaseSurveillanceModel_' in m else m for m in df_final.loc[:,'MODEL_NAME']]

In [62]:
df_final.loc[:,'MODEL_NAME'].value_counts(dropna=False)

MODEL_NAME
General                               3192
STD                                   1221
Childhood_Lead_Child_Model             361
Hep                                    296
ClusterModel                           195
Childhood_Lead_Investigation_Model     125
PortalApplicationModel                  30
Coinfection                             10
EHRAndScreeningModel_STD                 8
IsolationAndControlModel                 4
AggNETSS                                 2
Name: count, dtype: int64

In [104]:
#df_final[df_final['target_field'].isna()]

In [63]:
df_final.to_csv('output/maven_HealthcareUtilization_origami_similarity_match.csv',index=False)